#THIS WHOLE SCRIPT IS FOR GOOGLE EARTH ENGINE WORKFLOW IN PYTHON API

In [ ]:
#import packages to run GEE python API

import ee #for google earth engine python api
import ipyleaflet #for GEE like interactive maps in python
import geemap  #for GEE like interactive maps in python
import pprint #for printing complex data structure with proper identation and line breaks
import time #for pausing, timing and timestamps

#import other packages for geospatial analysis

import os
import numpy as np
import pandas as pd
import os #for interacting with operating system
import re #regular expresssion -- for advanced string pattern matchin
import glob #for finding files using wildcards
import numpy as np #for arrays
import pandas as pd #for tabular data
import xarray as xr #for multi-dimensinal labeled arrays
import geopandas as gpd #for vector spatial data
import rioxarray #for connecting xarryy and GIS
import regionmask #for masking of gridded data by regions

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library
ee.Initialize(project='ee-diptibaral21') #this will produce token in you GEE account 
#which you need to put in the link generated here so you can initialize your GEE account here

In [ ]:
# STEP 0: Setup and Counties

In [ ]:
#defining study area

counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_counties = ['Butte','Colusa','Glenn','Placer','Sacramento','San Joaquin',
                     'Sutter','Yolo','Yuba']

sac_valley = counties.filter(ee.Filter.eq('STATEFP', '06')) \
                     .filter(ee.Filter.inList('NAME', selected_counties))

#use "\" to indicate python that the line continues on the next line, otherwise it will 
#think the line ended there

In [ ]:
#visualizing Sac Valley Counties

Map = geemap.Map()
Map.centerObject(sac_valley, 8)  

# Add the layer
Map.addLayer(sac_valley, {},'Sacramento Valley Counties')
Map

In [ ]:
# STEP 1: Rice Masking

In [ ]:
#function that masks rice from all the crops in the Crop Data Layer (CDL)
    #this is needed because we want to extract climate variables only from rice 
    #growing pixels within each county and not the whole county area
    #rice is coded as 3 in cropland category in the CDL

def mask_rice(image): 
   
    """
    Masks rice pixels from a Crop Data Layer (CDL) image.

    For a given CDL image:
    - Selects the 'cropland' band.
    - Identifies rice pixels (coded as 3 in CDL).
    - Masks the image to keep only rice pixels.
    - Clips the image to the Sacramento Valley region (`sac_valley`).
    - Sets non-rice pixels to 0 (binary: 1 = rice, 0 = non-rice).
    - Preserves the original image's 'system:time_start' property to maintain temporal metadata.

    Parameters:
    ----------
    image : ee.Image
        A single-year CDL image with a 'cropland' band.

    Returns:
    -------
    ee.Image
        A binary image where rice pixels are 1, non-rice pixels are 0, clipped to the Sacramento Valley.
    """
    
    rice_mask = image.select('cropland').eq(3)
    return image.updateMask(rice_mask) \
    .clip(sac_valley) \
    .unmask(0) \
    .set('system:time_start', image.get('system:time_start'))


#this function filters the cdl from start to end year and for each year image applies rice mask, 
#then creates rice sum which is the no of years rice grown from start to end year and then creates 
#rice frequency which is 1 if rice is grown was grown for at least one year  -- we are interested in 
#rice frequency because while defining the rice growing area  the area with atleast 1 year of cropping 
#is defined as rice growing area for this research

def get_rice_frequency(start_year, end_year):
   
    """
    Calculates rice presence and frequency over multiple years from the CDL.

    For the specified year range:
    - Filters the CDL ImageCollection to the given date range.
    - Selects the 'cropland' band from each image.
    - Applies `mask_rice` to keep only rice pixels for each year.
    - Reduces the collection by summing rice presence across years (`rice_sum`).
      This gives the total number of years rice was grown at each pixel.
    - Creates `rice_frequency`, a binary image where 1 indicates rice was grown 
      at least once during the period, and 0 otherwise.

    Parameters:
    ----------
    start_year : int
        The first year to include in the analysis.
    end_year : int
        The last year to include in the analysis.

    Returns:
    -------
    rice_sum : ee.Image
        An image where each pixel value represents the number of years rice was grown (range: 0 to number of years).
    rice_frequency : ee.Image
        A binary image where 1 indicates rice was grown at least once, 0 otherwise.
    """
    cdl = ee.ImageCollection("USDA/NASS/CDL") \
            .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
            .map(lambda img: img.select(['cropland'])) \
            .map(mask_rice)

    rice_sum = cdl.reduce(ee.Reducer.sum())
    rice_frequency = rice_sum.gt(0)
    return rice_sum, rice_frequency
    
rice_sum, rice_frequency = get_rice_frequency(1997, 2023)   

Map.addLayer(rice_sum.clip(sac_valley),{
    'min':0, 
    'max': 27, 
    'palette': ['#e5f5e0', '#a1d99b', '#31a354']},'Rice Frequency (Years)')
Map
#palette from colorbrew

In [ ]:
# STEP 2: Processing loca Data - filtering from 2030-2100, converting k to C, keeping only tmmx, tmmn, tmean -- script (loca_county_daily_means.py)

In [ ]:
# STEP 3: Calculating GDD - loca_gdd.py

In [ ]:
# STEP 4: Phenology Detection Using Growing Degree Day Model - loca_growth_stages.ipynb 

In [ ]:
# STEP 5: Stress Calculation


# GEE-equivalent county/year/stage stress indices from LOCA2 NetCDFs
# - Reads 20 NetCDF files (10 models × 2 SSPs)
# - Vars: tmmn, tmmx, tmean (dims: time, lat, lon)
# - For each county × year × stage window:
#     * daily stress per pixel (booting/flowering logic)
#     * stage-window climate means (tmmn/tmmx/tmean)
#     * stage-window stress sums (cdstress_bo/cdstress_fl/htstress_fl)
#     * county reduction = AREA-WEIGHTED mean using cos(lat) weights (like equal-area intent in GEE)
# - Exports one CSV per NetCDF file


In [ ]:

#here we will calculate stress indices based on the above formula for each pixel -- then aggregate those pixel values over each 
#county to get the county mean stress for each year

#defining the thresholds

booting_cold = 13
flowering_cold = 10.9
flowering_heat = 37.5


def calculate_daily_stress(ds: xr.Dataset, stage: str) -> xr.Dataset:

    """
    Calculate daily temperature stress indices for rice based on phenological stage.

    This function computes pixel-level daily cold and/or heat stress using
    stage-specific temperature thresholds from Espe et al. (2017). Stress is
    calculated for *all dates*, but only the indices relevant to the specified
    growth stage are populated; other stress variables remain zero and can be
    masked later using stage-specific calendars.

    Cold stress is defined as the positive difference between a critical
    temperature threshold and daily minimum temperature. Heat stress is defined
    as the positive difference between daily maximum temperature and a critical
    heat threshold.

    Parameters
    ----------
    ds : xr.Dataset
        Xarray Dataset containing daily temperature variables with dimensions
        (time, lat, lon) or equivalent. Must include:
        - tasmin : daily minimum temperature (°C)
        - tasmax : daily maximum temperature (°C)
        - tmean  : daily mean temperature (°C)

    stage : str
        Rice phenological stage for which stress should be computed.
        Supported values are:
        - "booting"   : cold stress during booting stage
        - "flowering" : cold and heat stress during flowering stage
        - "grain_fill": no temperature stress applied (all stress indices = 0)

    Returns
    -------
    xr.Dataset
        Dataset containing the original temperature variables and the following
        stress indices (all aligned to the input dimensions and coordinates):

        - cdstress_bo : Cold stress during booting stage (°C·day)
        - cdstress_fl : Cold stress during flowering stage (°C·day)
        - htstress_fl : Heat stress during flowering stage (°C·day)

        Stress variables not relevant to the selected stage are returned as
        zero-filled arrays to ensure consistent structure across stages.

    Notes
    -----
    - Thresholds are fixed and based on Espe et al. (2017):
        * Booting cold threshold: 13.0 °C
        * Flowering cold threshold: 10.9 °C
        * Flowering heat threshold: 37.5 °C
    - Stress indices are computed per pixel and per day and are intended to be
      aggregated temporally and spatially (e.g., county means) in downstream
      analysis.
    - xr.zeros_like is used to preserve metadata, coordinates, and dimensions.

    Raises
    ------
    ValueError
        If `stage` is not one of the supported growth stages.
    """    
        
    #for loca data tmin and tmax are named as tasmin and tasmax
    tmmn = ds["tasmin"]  
    tmmx = ds["tasmax"]
    tmean = ds["tmean"]


    # ALWAYS define these so they're never missing
    #xr.zero_like(other_array) -- it creats a zero-filled array that copies the shape, 
    #dimensions, and coords of the existing 
    #DataArray or Dataset -- np.zeros(da.shape) looses metadata no dims, no coords, 
    #no labels so xr.zerolike()
    #first we create dataset for stress indices similar to tmmn, tmmx with all 0, 
    #then later add the stress indices values
    #specific to stages define by if statements below
    
    cdstress_bo = xr.zeros_like(tmmn).rename("cdstress_bo")
    cdstress_fl = xr.zeros_like(tmmn).rename("cdstress_fl")
    htstress_fl = xr.zeros_like(tmmx).rename("htstress_fl")

    if stage == "booting":
        cdstress_bo = xr.where(
            tmmn < booting_cold,
            (booting_cold - tmmn),
            0.0
        ).rename("cdstress_bo")
        
        #renaning is important here because we are using tmmn band here so xarrya takes 
        #the name form tmmn - but it is no longer temp-its cold stress, although we name 
        #the variable cold stress, we need to give the new band cold stress name

    elif stage == "flowering":
        cdstress_fl = xr.where(
            tmmn < flowering_cold,
            (flowering_cold - tmmn),
            0.0
        ).rename("cdstress_fl")

        htstress_fl = xr.where(
            tmmx > flowering_heat,
            (tmmx - flowering_heat),
            0.0
        ).rename("htstress_fl")

    elif stage == "grain_fill":
        # keep zeros 
        pass

    else:
        raise ValueError(f"Unknown stage: {stage}")

    return xr.Dataset(
        {
            "tmmn": tmmn,
            "tmmx": tmmx,
            "tmean": tmean,
            "cdstress_bo": cdstress_bo,
            "cdstress_fl": cdstress_fl,
            "htstress_fl": htstress_fl,   
        }
    )


In [ ]:
#County masks + area weights (lat/lon)
    #convert counties into a maskable region object
    #this function is to build a mask using regionmask.Regions which assign each grid cell in a raset (here LOCA) 
    #to a region(county) and creats masks like which pixels belong to which county
    #this is useful for aggregating LOCA data by county -- in GEE reduce by regions is used

def build_county_mask(counties_gdf: gpd.GeoDataFrame, name_col: str = "NAME"):
    """
    Create a regionmask Regions object from a counties GeoDataFrame.

    This function reprojects the input GeoDataFrame to WGS84 (EPSG:4326)
    and constructs a `regionmask.Regions` object using each county's
    geometry as a separate region. County names are used both as region
    names and abbreviations.

    Parameters
    ----------
    counties_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing county polygon geometries.
    name_col : str, default "NAME"
        Column in `counties_gdf` that contains county names to label
        each region.

    Returns
    -------
    regionmask.Regions
        A reusable Regions object that can be applied to gridded data
        (e.g., xarray datasets) to generate county-level masks.
    """
    gdf = counties_gdf.to_crs("EPSG:4326").copy()

    regions = regionmask.Regions(
        outlines=list(gdf.geometry),  #list of polygon shapes (one per county)
        names=list(gdf[name_col].astype(str).values), #name for each county
        abbrevs=list(gdf[name_col].astype(str).values), #abbreviation used internally by regionmask here they are same as names but sometimes people use codes like FIPS
        name="counties", #labels the whole collection as counties
    )
    return regions #returns reuseable object

In [ ]:

#this fuction calculates county-level yearly stress indices for specific rice growth stage
# this is the Python/xarry equivalent of Google Earth Engine reducer pipeline:
#EE image collection -- time filter -- per-pixel stress - reduce over time-- reduce over region

def calculate_yearly_stage_stress(stage_feature,
                                  growing_season_images: xr.Dataset,
                                  regions: regionmask.Regions,
                                  mask: xr.DataArray,
                                  weights: xr.DataArray,
                                  name_col: str = "NAME"):
    """
    Python equivalent of EE pipeline.

    Parameters
    ----------
    stage_feature : dict-like or pd.Series
        Must have keys: 'county', 'year', 'stage', 'start_date', 'end_date'
    growing_season_images : xr.Dataset
        Must have vars: 'tmmn','tmmx' (optional 'tmean') and coords 'time'
        Spatial coords should match what mask/weights were built on.
    regions : regionmask.Regions
        Regions created from county polygons (EPSG:4326) using build_county_mask.
    mask : xr.DataArray
        County id mask on the dataset grid (e.g., regions.mask(ds['lon'], ds['lat']))
    weights : xr.DataArray
        Area weights on the grid (e.g., cos(lat) for lat/lon grid), broadcastable to (lat,lon)

    Returns
    -------
    pd.DataFrame
        One-row dataframe with:
        county, stage, year, tmmn, tmmx, tmean, cdstress_bo, cdstress_fl, htstress_fl
        where climate vars are time-means and stress vars are time-sums,
        all spatially averaged over the county polygon.
    """

    #parse stage feature
    county = str(stage_feature["county"])
    year   = int(stage_feature["year"])
    stage  = str(stage_feature["stage"])
    start  = pd.to_datetime(stage_feature["start_date"])
    end    = pd.to_datetime(stage_feature["end_date"])

    # county id in the regionmask object 
        # Map the county name to its corresponding numeric region ID in the RegionMask object.
        # This is needed because spatial data arrays use numeric IDs to identify regions,
        # not names. We first create a dictionary mapping names to IDs, check if the
        # requested county exists, and then retrieve its ID (rid) for use in masking
        # or extracting data specific to that county.
    
    name_to_id = {n: i for i, n in enumerate(regions.names)}
    if county not in name_to_id:
        raise ValueError(f"County '{county}' not found in regions.names.")
    rid = name_to_id[county]

    #subset dataset by stage window 
    ds_sub = growing_season_images.sel(time=slice(start, end))

    # daily stress  
    daily_stress = calculate_daily_stress(ds_sub, stage)  

    # time aggregation (matches EE)
    climate_means = daily_stress[["tmmn", "tmmx", "tmean"]].mean("time", skipna=True)
    stress_sums   = daily_stress[["cdstress_bo", "cdstress_fl", "htstress_fl"]].sum("time", skipna=True)

    yearly = xr.merge([climate_means, stress_sums])

    # spatial reduction by county (mean over pixels inside county) 
        # Compute the county-level average for each variable:
        # 1. Mask pixels outside the target county using mask == rid, so only pixels
        #    inside the county contribute to the calculation.
        # 2. Use a weighted mean across spatial dimensions (lat/lon) to account for
        #    differences in pixel area or grid distortion (e.g., higher latitude pixels
        #    cover smaller physical area). This ensures the county average reflects
        #    the true area-weighted value, not just a simple pixel mean.
        # 3. skipna=True ignores missing or masked pixels outside the county.

    out = {}
    for var in yearly.data_vars:
        da = yearly[var].where(mask == rid)
        # weights expected to be compatible with da's spatial dims ( lat/lon)
        out[var] = float(da.weighted(weights).mean(dim=[d for d in da.dims if d != "time"], skipna=True).values)

    # output
    out_row = {
        "county": county,
        "stage": stage,
        "year": year,
        "tmmn": out.get("tmmn", np.nan),
        "tmmx": out.get("tmmx", np.nan),
        "tmean": out.get("tmean", np.nan),
        "cdstress_bo": out.get("cdstress_bo", np.nan),
        "cdstress_fl": out.get("cdstress_fl", np.nan),
        "htstress_fl": out.get("htstress_fl", np.nan),
    }

    return pd.DataFrame([out_row])


In [ ]:

# paths

CLIMATE_DIR = os.environ.get("CLIMATE_DIR")

stage_table_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/LOCA_stage_table")
out_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/LOCA_stage_stress_tables")
loca_netcdf_dir = os.path.join(CLIMATE_DIR, "run_project/input_data/loca2")
shapefile = os.path.join(CLIMATE_DIR, "run_project/input_data/shape_files/CA_9counties_shapefile.shp")



# counties -- regions
counties_gdf = gpd.read_file(shapefile).to_crs("EPSG:4326")
regions = build_county_mask(counties_gdf)


# Helper function top load matching LOCA dataset


def load_loca_dataset(stage_table_file, netcdf_dir):
    """
    Matches stage table:
      ACCESS-CM2_ssp245_r1i1p1f1__stage_table.csv
    to NetCDF:
      ACCESS-CM2_ssp245_r1i1p1f1_rice_temp.nc
    """
    base = os.path.basename(stage_table_file)
    model = base.split("_ssp")[0]

    m = re.search(r"(ssp\d{3})", base)
    if not m:
        raise ValueError(f"Could not parse SSP from {base}")
    ssp = m.group(1)

    nc_path = os.path.join(netcdf_dir, f"{model}_{ssp}_r1i1p1f1_rice_temp.nc")
    if not os.path.exists(nc_path):
        raise FileNotFoundError(f"Missing NetCDF:\n{nc_path}")

    print(f"[load_loca_dataset] Using: {nc_path}")
    ds = xr.open_dataset(nc_path)

    # converting Kelvin to Celsius
        #in loca netcdf file, temps are in K so change to C
    u = str(ds["tasmin"].attrs.get("units", "")).lower()
        # .attrs to get the metadata, .get to access the valye for the key units (its in dictionary form )
        # usging .attrs['units'] won't work because the metadata is in dict form
    vmin = float(ds["tasmin"].min())
    if ("k" in u) or (vmin > 100):
        ds = ds.copy()
        ds["tasmin"] = ds["tasmin"] - 273.15
        ds["tasmax"] = ds["tasmax"] - 273.15
        ds["tasmin"].attrs["units"] = "degC"
        ds["tasmax"].attrs["units"] = "degC"

    # renaming the coords names for regionmask
    rename = {}
    if "longitude" in ds.coords and "lon" not in ds.coords:
        rename["longitude"] = "lon"
    if "latitude" in ds.coords and "lat" not in ds.coords:
        rename["latitude"] = "lat"
    if rename:
        ds = ds.rename(rename)

    return ds

    ds = kelvin_to_celsius(ds)

# MAIN LOOP (20 tables)

stage_tables = sorted(glob.glob(os.path.join(stage_table_dir, "*_stage_table.csv")))
    #this gives the stage tables paths

for st_path in stage_tables:
    tag = os.path.basename(st_path).replace("_stage_table.csv", "")
    print(f"\nProcessing: {tag}")

    # load stage table
    stage_table_df = pd.read_csv(st_path)
    stage_table_df["start_date"] = pd.to_datetime(stage_table_df["start_date"])
    stage_table_df["end_date"]   = pd.to_datetime(stage_table_df["end_date"])
    stage_table_df["year"]       = stage_table_df["year"].astype(int)

    # load matching LOCA data
    ds = load_loca_dataset(st_path, loca_netcdf_dir)

    # build mask + weights for THIS grid
        # Build a region mask that assigns each grid cell to a county,
        # and latitude-based area weights (cos(lat)) to account for decreasing
        # grid-cell area toward higher latitudes when computing spatial averages
    
    mask = regions.mask(ds["lon"], ds["lat"])          # (lat, lon) -- always remember x is lon and y is lat
    weights = np.cos(np.deg2rad(ds["lat"]))             # latitude weights

    # This step is equivalent to stage_table.map(...).flatten() in GEE

    #Initialize an empty list to store results
    rows = []
    #loop over each row of the df - each row represents a stage feature for a specific county and year
    for idx, row in stage_table_df.iterrows():
        #convert the row to a dictionary and calculate stress indices for that row

        out = calculate_yearly_stage_stress(
            stage_feature=row.to_dict(), # the county/year/stage info
            growing_season_images=ds, # climate data (xr.Dataset) for the growing season
            regions=regions, #county polygons 
            mask=mask, # mask to select only rice pixels
            weights=weights #pixel area weights
        )
        rows.append(out)

    if len(rows) == 0:
        print(f"No stress calculated for {tag}")
        all_year_table = pd.DataFrame(columns=["county", "year", "stage", "cdstress_bo", "cdstress_fl", "htstress_fl"])
    else:
        all_year_table = pd.concat(rows, ignore_index=True)
        #if its an empty row, it will give empty df with heading as defined, if not then it will be concatenated to the rows[]. 
        #its good idea to create the empty df to check if the rows are empty or not

    # selecte only the columns needed
    all_year_table_selected = all_year_table[
        ["county", "year", "stage",
         "cdstress_bo", "cdstress_fl", "htstress_fl"]
    ]

    # checking the first feature 
    first_export_feature = all_year_table_selected.iloc[0].to_dict()
    print("First feature:", first_export_feature)

    # save
    out_csv = os.path.join(out_dir, f"{tag}_yearly_stage_stress.csv")
    all_year_table_selected.to_csv(out_csv, index=False)
    print(f"Saved  {out_csv}")

